# Black Grouse 2030 Land-Use Scenario Project

## Stage 5: Restoration Suitability

This notebook prepares the spatial layers required to simulate alternative 2030 restoration scenarios.

Restoration candidates are restricted to agricultural-matrix pixels in the 2024 land-use raster. Forest, existing core habitat, urban and infrastructure, water, and pixels outside the study area are excluded.

The notebook calculates:

- distance to existing core habitat;
- distance to urban and infrastructure;
- surrounding core-habitat amount;
- surrounding urban and infrastructure amount;
- a combined restoration-suitability score.

The scenario simulations themselves will be created in Stage 6.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio

from scipy import ndimage

print("Packages imported successfully.")

Packages imported successfully.


In [2]:
PROJECT_DIR = Path.cwd()

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
TABLES_DIR = PROJECT_DIR / "outputs" / "tables"

LANDUSE_2024_PATH = (
    PROCESSED_DIR / "LGN2024_harmonised_25m.tif"
)

RESTORATION_CLASS = 1
CORE_HABITAT_CLASS = 2
URBAN_CLASS = 4

print(
    "2024 land-use raster:",
    "FOUND" if LANDUSE_2024_PATH.exists() else "MISSING",
)

print(LANDUSE_2024_PATH)

2024 land-use raster: FOUND
C:\Users\smit1\BlackGrouse_2030\data\processed\LGN2024_harmonised_25m.tif


In [3]:
with rasterio.open(LANDUSE_2024_PATH) as src:

    landuse_2024 = src.read(1)

    raster_profile = src.profile.copy()
    raster_transform = src.transform
    raster_crs = src.crs
    pixel_size_metres = abs(src.res[0])


analysis_mask = landuse_2024 != 0
candidate_mask = landuse_2024 == RESTORATION_CLASS
core_habitat_mask = landuse_2024 == CORE_HABITAT_CLASS
urban_mask = landuse_2024 == URBAN_CLASS


pixel_area_km2 = (
    pixel_size_metres
    * pixel_size_metres
    / 1_000_000
)


print(f"Shape: {landuse_2024.shape}")
print(f"CRS: {raster_crs}")
print(f"Pixel size: {pixel_size_metres:.0f} m")

print(
    "Analysis area:",
    f"{analysis_mask.sum() * pixel_area_km2:.2f} km²",
)

print(
    "Existing core habitat:",
    f"{core_habitat_mask.sum() * pixel_area_km2:.2f} km²",
)

print(
    "Eligible agricultural matrix:",
    f"{candidate_mask.sum() * pixel_area_km2:.2f} km²",
)

print(
    "Excluded area:",
    f"{((analysis_mask) & (~candidate_mask)).sum() * pixel_area_km2:.2f} km²",
)

Shape: (1592, 1023)
CRS: EPSG:28992
Pixel size: 25 m
Analysis area: 762.88 km²
Existing core habitat: 54.41 km²
Eligible agricultural matrix: 456.27 km²
Excluded area: 306.61 km²


In [4]:
distance_to_core_metres = ndimage.distance_transform_edt(
    ~core_habitat_mask,
    sampling=pixel_size_metres,
).astype(np.float32)

distance_to_urban_metres = ndimage.distance_transform_edt(
    ~urban_mask,
    sampling=pixel_size_metres,
).astype(np.float32)

distance_to_core_metres[~analysis_mask] = np.nan
distance_to_urban_metres[~analysis_mask] = np.nan


candidate_core_distances = distance_to_core_metres[
    candidate_mask
]

candidate_urban_distances = distance_to_urban_metres[
    candidate_mask
]


print("Distance to existing core habitat for candidate pixels:")
print(
    f"Minimum: {np.nanmin(candidate_core_distances):.0f} m"
)
print(
    f"Median: {np.nanmedian(candidate_core_distances):.0f} m"
)
print(
    f"Maximum: {np.nanmax(candidate_core_distances):.0f} m"
)

print("\nDistance to urban and infrastructure for candidate pixels:")
print(
    f"Minimum: {np.nanmin(candidate_urban_distances):.0f} m"
)
print(
    f"Median: {np.nanmedian(candidate_urban_distances):.0f} m"
)
print(
    f"Maximum: {np.nanmax(candidate_urban_distances):.0f} m"
)

Distance to existing core habitat for candidate pixels:
Minimum: 25 m
Median: 326 m
Maximum: 2000 m

Distance to urban and infrastructure for candidate pixels:
Minimum: 25 m
Median: 100 m
Maximum: 633 m


In [5]:
DISTANCE_TO_CORE_PATH = (
    PROCESSED_DIR / "distance_to_core_habitat_2024_m.tif"
)

DISTANCE_TO_URBAN_PATH = (
    PROCESSED_DIR / "distance_to_urban_2024_m.tif"
)

FLOAT_NODATA = -9999.0


def save_float_raster(
    output_path,
    raster_array,
):
    output_profile = raster_profile.copy()

    output_profile.update(
        {
            "driver": "GTiff",
            "dtype": "float32",
            "count": 1,
            "nodata": FLOAT_NODATA,
            "compress": "lzw",
        }
    )

    output_array = np.where(
        np.isfinite(raster_array),
        raster_array,
        FLOAT_NODATA,
    ).astype(np.float32)

    with rasterio.open(
        output_path,
        "w",
        **output_profile,
    ) as destination:
        destination.write(output_array, 1)

    print(f"Saved: {output_path}")


save_float_raster(
    DISTANCE_TO_CORE_PATH,
    distance_to_core_metres,
)

save_float_raster(
    DISTANCE_TO_URBAN_PATH,
    distance_to_urban_metres,
)

Saved: C:\Users\smit1\BlackGrouse_2030\data\processed\distance_to_core_habitat_2024_m.tif
Saved: C:\Users\smit1\BlackGrouse_2030\data\processed\distance_to_urban_2024_m.tif


In [6]:
from scipy.signal import fftconvolve


NEIGHBOURHOOD_RADIUS_METRES = 1000

radius_pixels = int(
    NEIGHBOURHOOD_RADIUS_METRES / pixel_size_metres
)

row_offsets, column_offsets = np.ogrid[
    -radius_pixels:radius_pixels + 1,
    -radius_pixels:radius_pixels + 1,
]

circular_kernel = (
    row_offsets**2 + column_offsets**2
    <= radius_pixels**2
).astype(np.float32)


valid_neighbourhood_pixels = fftconvolve(
    analysis_mask.astype(np.float32),
    circular_kernel,
    mode="same",
)

core_neighbourhood_pixels = fftconvolve(
    core_habitat_mask.astype(np.float32),
    circular_kernel,
    mode="same",
)

urban_neighbourhood_pixels = fftconvolve(
    urban_mask.astype(np.float32),
    circular_kernel,
    mode="same",
)


with np.errstate(
    divide="ignore",
    invalid="ignore",
):

    surrounding_core_proportion = (
        core_neighbourhood_pixels
        / valid_neighbourhood_pixels
    )

    surrounding_urban_proportion = (
        urban_neighbourhood_pixels
        / valid_neighbourhood_pixels
    )


surrounding_core_proportion[
    ~analysis_mask
] = np.nan

surrounding_urban_proportion[
    ~analysis_mask
] = np.nan


candidate_core_proportion = surrounding_core_proportion[
    candidate_mask
]

candidate_urban_proportion = surrounding_urban_proportion[
    candidate_mask
]


print("Candidate-pixel surrounding core-habitat proportion:")
print(
    f"Minimum: {np.nanmin(candidate_core_proportion):.3f}"
)
print(
    f"Median: {np.nanmedian(candidate_core_proportion):.3f}"
)
print(
    f"Maximum: {np.nanmax(candidate_core_proportion):.3f}"
)

print("\nCandidate-pixel surrounding urban proportion:")
print(
    f"Minimum: {np.nanmin(candidate_urban_proportion):.3f}"
)
print(
    f"Median: {np.nanmedian(candidate_urban_proportion):.3f}"
)
print(
    f"Maximum: {np.nanmax(candidate_urban_proportion):.3f}"
)

Candidate-pixel surrounding core-habitat proportion:
Minimum: -0.000
Median: 0.012
Maximum: 0.699

Candidate-pixel surrounding urban proportion:
Minimum: 0.003
Median: 0.083
Maximum: 0.895


In [7]:
SURROUNDING_CORE_PATH = (
    PROCESSED_DIR
    / "surrounding_core_habitat_1km_proportion.tif"
)

SURROUNDING_URBAN_PATH = (
    PROCESSED_DIR
    / "surrounding_urban_1km_proportion.tif"
)


save_float_raster(
    SURROUNDING_CORE_PATH,
    surrounding_core_proportion,
)

save_float_raster(
    SURROUNDING_URBAN_PATH,
    surrounding_urban_proportion,
)

Saved: C:\Users\smit1\BlackGrouse_2030\data\processed\surrounding_core_habitat_1km_proportion.tif
Saved: C:\Users\smit1\BlackGrouse_2030\data\processed\surrounding_urban_1km_proportion.tif


In [8]:
def robust_scale(
    raster_array,
    candidate_pixels,
    higher_is_better=True,
):
    """
    Scale candidate-pixel values from 0 to 1 using the
    2nd and 98th percentiles to reduce the effect of outliers.
    """

    candidate_values = raster_array[candidate_pixels]

    lower_limit = np.nanpercentile(
        candidate_values,
        2,
    )

    upper_limit = np.nanpercentile(
        candidate_values,
        98,
    )

    scaled_array = (
        raster_array - lower_limit
    ) / (
        upper_limit - lower_limit
    )

    scaled_array = np.clip(
        scaled_array,
        0,
        1,
    )

    if not higher_is_better:
        scaled_array = 1 - scaled_array

    scaled_array[~candidate_pixels] = np.nan

    return scaled_array.astype(np.float32)


# Shorter distance to existing core habitat is preferable
core_proximity_score = robust_scale(
    distance_to_core_metres,
    candidate_mask,
    higher_is_better=False,
)

# Greater distance from urban and infrastructure is preferable
urban_distance_score = robust_scale(
    distance_to_urban_metres,
    candidate_mask,
    higher_is_better=True,
)

# More existing core habitat within 1 km is preferable
surrounding_core_score = robust_scale(
    surrounding_core_proportion,
    candidate_mask,
    higher_is_better=True,
)

# Less urban and infrastructure within 1 km is preferable
low_urban_score = robust_scale(
    surrounding_urban_proportion,
    candidate_mask,
    higher_is_better=False,
)


# Equal weighting of the four suitability components
restoration_suitability_score = (
    0.25 * core_proximity_score
    + 0.25 * urban_distance_score
    + 0.25 * surrounding_core_score
    + 0.25 * low_urban_score
).astype(np.float32)


candidate_scores = restoration_suitability_score[
    candidate_mask
]


print("Restoration-suitability score for candidate pixels:")
print(f"Minimum: {np.nanmin(candidate_scores):.3f}")
print(f"Median: {np.nanmedian(candidate_scores):.3f}")
print(f"Mean: {np.nanmean(candidate_scores):.3f}")
print(f"Maximum: {np.nanmax(candidate_scores):.3f}")

Restoration-suitability score for candidate pixels:
Minimum: 0.000
Median: 0.461
Mean: 0.468
Maximum: 1.000


In [9]:
SUITABILITY_RASTERS = {
    "core_proximity_score":
        core_proximity_score,

    "urban_distance_score":
        urban_distance_score,

    "surrounding_core_score":
        surrounding_core_score,

    "low_urban_score":
        low_urban_score,

    "restoration_suitability_score":
        restoration_suitability_score,
}


for raster_name, raster_array in SUITABILITY_RASTERS.items():

    output_path = (
        PROCESSED_DIR / f"{raster_name}.tif"
    )

    save_float_raster(
        output_path,
        raster_array,
    )


suitability_summary = pd.DataFrame(
    {
        "variable": [
            "core_proximity_score",
            "urban_distance_score",
            "surrounding_core_score",
            "low_urban_score",
            "restoration_suitability_score",
        ],
        "minimum": [
            np.nanmin(core_proximity_score),
            np.nanmin(urban_distance_score),
            np.nanmin(surrounding_core_score),
            np.nanmin(low_urban_score),
            np.nanmin(restoration_suitability_score),
        ],
        "median": [
            np.nanmedian(core_proximity_score),
            np.nanmedian(urban_distance_score),
            np.nanmedian(surrounding_core_score),
            np.nanmedian(low_urban_score),
            np.nanmedian(restoration_suitability_score),
        ],
        "mean": [
            np.nanmean(core_proximity_score),
            np.nanmean(urban_distance_score),
            np.nanmean(surrounding_core_score),
            np.nanmean(low_urban_score),
            np.nanmean(restoration_suitability_score),
        ],
        "maximum": [
            np.nanmax(core_proximity_score),
            np.nanmax(urban_distance_score),
            np.nanmax(surrounding_core_score),
            np.nanmax(low_urban_score),
            np.nanmax(restoration_suitability_score),
        ],
    }
).round(3)


SUITABILITY_SUMMARY_PATH = (
    TABLES_DIR / "restoration_suitability_summary.csv"
)

suitability_summary.to_csv(
    SUITABILITY_SUMMARY_PATH,
    index=False,
)

display(suitability_summary)

print("\nSummary saved:")
print(SUITABILITY_SUMMARY_PATH)

Saved: C:\Users\smit1\BlackGrouse_2030\data\processed\core_proximity_score.tif
Saved: C:\Users\smit1\BlackGrouse_2030\data\processed\urban_distance_score.tif
Saved: C:\Users\smit1\BlackGrouse_2030\data\processed\surrounding_core_score.tif
Saved: C:\Users\smit1\BlackGrouse_2030\data\processed\low_urban_score.tif
Saved: C:\Users\smit1\BlackGrouse_2030\data\processed\restoration_suitability_score.tif


,variable,minimum,median,mean,maximum
0,core_proximity_score,0.0,0.692,0.648,1.0
1,urban_distance_score,0.0,0.242,0.289,1.0
2,surrounding_core_score,0.0,0.049,0.155,1.0
3,low_urban_score,0.0,0.849,0.778,1.0
4,restoration_suitability_score,0.0,0.461,0.468,1.0



Summary saved:
C:\Users\smit1\BlackGrouse_2030\outputs\tables\restoration_suitability_summary.csv


In [10]:
SUITABILITY_OUTPUT_PATHS = {
    raster_name: PROCESSED_DIR / f"{raster_name}.tif"
    for raster_name in SUITABILITY_RASTERS
}

validation_records = []

for raster_name, raster_path in SUITABILITY_OUTPUT_PATHS.items():

    with rasterio.open(raster_path) as src:

        raster_data = src.read(1)
        nodata_value = src.nodata

        valid_values = raster_data[
            raster_data != nodata_value
        ]

        valid_mask = raster_data != nodata_value

        validation_records.append(
            {
                "raster": raster_name,
                "file_exists": raster_path.exists(),
                "crs": str(src.crs),
                "width": src.width,
                "height": src.height,
                "resolution": src.res,
                "valid_pixels": int(valid_mask.sum()),
                "expected_candidate_pixels": int(
                    candidate_mask.sum()
                ),
                "same_candidate_mask": np.array_equal(
                    valid_mask,
                    candidate_mask,
                ),
                "minimum": round(
                    float(valid_values.min()),
                    3,
                ),
                "maximum": round(
                    float(valid_values.max()),
                    3,
                ),
            }
        )


suitability_validation = pd.DataFrame(
    validation_records
)

VALIDATION_OUTPUT_PATH = (
    TABLES_DIR / "restoration_suitability_validation.csv"
)

suitability_validation.to_csv(
    VALIDATION_OUTPUT_PATH,
    index=False,
)

display(suitability_validation)


all_checks_passed = (
    suitability_validation["file_exists"].all()
    and suitability_validation["same_candidate_mask"].all()
    and (
        suitability_validation["crs"]
        == "EPSG:28992"
    ).all()
    and (
        suitability_validation["minimum"] >= 0
    ).all()
    and (
        suitability_validation["maximum"] <= 1
    ).all()
)


if not all_checks_passed:
    raise ValueError(
        "One or more restoration-suitability checks failed."
    )


print("\nAll restoration-suitability checks passed.")

print("\nValidation table saved:")
print(VALIDATION_OUTPUT_PATH)

,raster,file_exists,crs,width,height,resolution,valid_pixels,expected_candidate_pixels,same_candidate_mask,minimum,maximum
0,core_proximity_score,True,EPSG:28992,1023,1592,"(25.0, 25.0)",730030,730030,True,0.0,1.0
1,urban_distance_score,True,EPSG:28992,1023,1592,"(25.0, 25.0)",730030,730030,True,0.0,1.0
2,surrounding_core_score,True,EPSG:28992,1023,1592,"(25.0, 25.0)",730030,730030,True,0.0,1.0
3,low_urban_score,True,EPSG:28992,1023,1592,"(25.0, 25.0)",730030,730030,True,0.0,1.0
4,restoration_suitability_score,True,EPSG:28992,1023,1592,"(25.0, 25.0)",730030,730030,True,0.0,1.0



All restoration-suitability checks passed.

Validation table saved:
C:\Users\smit1\BlackGrouse_2030\outputs\tables\restoration_suitability_validation.csv
